# Samatrix Practice: Student Placement Prediction

This notebook is designed to help you learn the ML workflow, not just submit a file.

This guided ML lab helps you practice:

- understanding the dataset
- selecting useful features
- validation
- encoding
- baseline modeling
- ROC-AUC evaluation
- experiments
- submission creation
- reflection

## Learning workflow

Observe -> Think -> Experiment -> Validate -> Improve -> Submit -> Reflect

The goal is to understand the ML process, not only generate a file.

## Step 1: Download the challenge files

Run the next cell first.

In [ ]:
#@title Step 1 : Download challenge files { display-mode: "form" }
#@markdown Run this cell once. You do not need to edit the code.

import json
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://practice.samatrix.io"
CHALLENGE_SLUG = "student-placement-001"
FORCE_DOWNLOAD = False

DATA_DIR = Path(".")
CONFIG_URL = f"{BASE_URL}/api/v1/challenges/{CHALLENGE_SLUG}/client-config"

print("Connecting to Samatrix Practice...")
response = requests.get(CONFIG_URL, timeout=30)

if response.status_code != 200:
    raise RuntimeError(f"Could not load challenge config: {response.status_code} {response.text[:300]}")

config = response.json()

files_to_download = {
    "train": "train.csv",
    "test": "test.csv",
    "sample_submission": "sample_submission.csv",
}

for asset_key, filename in files_to_download.items():
    path = DATA_DIR / filename

    if path.exists() and not FORCE_DOWNLOAD:
        print(f"Using existing {filename}")
        continue

    file_url = config["assets"][asset_key]
    file_response = requests.get(file_url, timeout=60)

    if file_response.status_code != 200:
        raise RuntimeError(f"Could not download {filename}: {file_response.status_code}")

    path.write_bytes(file_response.content)
    print(f"Downloaded {filename}")

print()
print("Challenge:", config.get("title"))
print("Metric:", config.get("task", {}).get("metric_name"))
print("Files are ready.")


## Step 2: Load the data

Run this cell to load:

- train.csv
- test.csv
- sample_submission.csv
- challenge configuration

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

train.head()


## 3. Understand the dataset

Inspect:

- rows and columns
- target column
- ID column
- numeric features
- categorical features
- missing values
- target balance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

task = config.get("task", {})

target_column = task.get(
    "target_column",
    "placed"
)

id_column = task.get(
    "id_column",
    "id"
)

prediction_column = task.get(
    "prediction_column",
    "prediction"
)

print("Target:", target_column)
print("ID:", id_column)

print("Train:", train.shape)
print("Test:", test.shape)

display(train.head())

## Feature Selection

IDs identify rows. They usually should not be used as prediction features.

In [ ]:
# TODO: Review candidate_features and remove any feature you believe is weak, noisy, or risky.
candidate_features = [
    c for c in train.columns
    if c not in [
        target_column,
        id_column
    ]
]

feature_columns = candidate_features.copy()

print(
    feature_columns
)


## Checkpoint 1: Dataset observations

In [ ]:
# TODO: Replace the placeholder answers below with your own dataset observations.
checkpoint_1_dataset_observations = {
    "target_column": target_column,
    "id_column": id_column,
    "why_id_should_not_be_used": "TODO",
    "features_noticed": "TODO",
    "missing_values": "TODO",
}

checkpoint_1_dataset_observations


## Validation Split

In [ ]:
from sklearn.model_selection import train_test_split


X = train[feature_columns]
y = train[target_column]


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(
    X_train.shape,
    X_valid.shape
)

## Encoding

In [ ]:
# TODO: Understand why one-hot encoding is needed before training the model.
X_train_model = pd.get_dummies(
    X_train,
    dummy_na=True
)

X_valid_model = pd.get_dummies(
    X_valid,
    dummy_na=True
)


X_valid_model = X_valid_model.reindex(
    columns=X_train_model.columns,
    fill_value=0
)

print(
    X_train_model.shape
)


## Baseline Model

## ML insight: probabilities vs hard labels

ROC-AUC evaluates ranking quality, so the model should usually submit probabilities rather than class labels.

hard labels are class outputs such as 0 or 1. Probabilities preserve more ranking information for ROC-AUC.

Use `predict_proba(... )[:, 1]` for the positive-class probability. Avoid using `predict(...)` for ROC-AUC submissions.


In [ ]:
# TODO: Confirm that this uses probabilities, not hard labels, for ROC-AUC.
# TODO: Try changing n_estimators, max_depth, or min_samples_leaf and compare validation ROC-AUC.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


model = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)


model.fit(
    X_train_model,
    y_train
)


valid_probabilities = model.predict_proba(
    X_valid_model
)[:, 1]


valid_auc = roc_auc_score(
    y_valid,
    valid_probabilities
)


print(
    "Validation ROC-AUC:",
    round(valid_auc, 4)
)


## Experiment

Try changing one thing:

- model parameters
- features
- preprocessing

Record what happened.

In [ ]:
# TODO: Record at least one experiment and whether validation ROC-AUC improved.
experiment_tracker = {
    "baseline_auc": float(valid_auc),
    "experiment": "TODO",
    "result": "TODO",
}

experiment_tracker


## Create predictions.csv

In [ ]:
X_full = train[feature_columns]
X_test = test[feature_columns]


X_full_model = pd.get_dummies(
    X_full,
    dummy_na=True
)

X_test_model = pd.get_dummies(
    X_test,
    dummy_na=True
)


X_test_model = X_test_model.reindex(
    columns=X_full_model.columns,
    fill_value=0
)


final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)


final_model.fit(
    X_full_model,
    y
)


predictions = final_model.predict_proba(
    X_test_model
)[:, 1]


submission = pd.DataFrame({
    id_column: test[id_column],
    prediction_column: predictions
})


submission.to_csv(
    "predictions.csv",
    index=False
)


print(
    "Created predictions.csv"
)

display(submission.head())

## Reflection

Write what you learned before submission.

In [ ]:
# TODO: Replace the placeholder answers below with your actual reflection before submitting.
student_reflection = {
    "best_model_choice": "TODO",
    "best_experiment": "TODO",
    "what_failed": "TODO",
    "next_improvement": "TODO",
}

student_reflection


## 4. Submit to Samatrix Practice

1. Generate Colab Submission Token.
2. Copy the token.
3. Run the Step 4 cell.
4. Paste the token.
5. Open Direct Review Room.

In [ ]:
#@title Step 4: Run Cell, Paste Token, and Submit { display-mode: "form" }

from pathlib import Path

import requests

EXPECTED_SUBMISSION_ENDPOINT_PATH = "/api/v1/submissions/attempt-token-submit"

submission_config = config.get("submission", {})
endpoint_path = submission_config.get("endpoint_path")

if endpoint_path != EXPECTED_SUBMISSION_ENDPOINT_PATH:
    raise RuntimeError(
        "Samatrix client-config returned an unexpected submission endpoint. "
        f"Expected {EXPECTED_SUBMISSION_ENDPOINT_PATH}, got {endpoint_path!r}."
    )

SUBMISSION_BASE_URL = config.get("base_url") or BASE_URL
SUBMISSION_URL = f"{SUBMISSION_BASE_URL}{endpoint_path}"


def submit_to_samatrix(colab_submission_token: str, predictions_path: str = "predictions.csv") -> dict | None:
    token = (colab_submission_token or "").strip()
    path = Path(predictions_path)

    if not token:
        print("❌ Colab Submission Token is required. Generate a token on the Samatrix challenge page and paste it here.")
        return None

    if not path.exists():
        print("❌ predictions.csv was not found. Run the model cell first.")
        return None

    try:
        with path.open("rb") as file_handle:
            response = requests.post(
                SUBMISSION_URL,
                headers={"Authorization": f"Bearer {token}"},
                files={"predictions_file": ("predictions.csv", file_handle, "text/csv")},
                timeout=60,
            )
    except requests.RequestException as exc:
        print("❌ Could not reach Samatrix Practice. Check your connection and try again.")
        print(str(exc))
        return None

    try:
        data = response.json()
    except ValueError:
        print(f"❌ Samatrix returned a non-JSON response with status {response.status_code}.")
        print(response.text[:1000])
        return None

    if response.status_code != 200:
        print(f"❌ Submission failed with status {response.status_code}.")
        print(data.get("detail") or json.dumps(data, indent=2))
        return data

    evaluation = data.get("evaluation") or {}

    print("✅ Submission completed.")
    print("Score:", evaluation.get("score_normalized"))
    print("Metric:", evaluation.get("metric_name"))
    print("Metric value:", evaluation.get("metric_value"))
    print("Review Room:", data.get("review_url") or data.get("review_full_url"))

    return data


try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    token_box = widgets.Password(
        description="Colab Submission Token:",
        placeholder="Paste token from Samatrix",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "120px"},
    )

    submit_button = widgets.Button(
        description="Open Direct Review Room",
        button_style="success",
        tooltip="Paste token and submit",
    )

    output = widgets.Output()

    def _on_submit_clicked(_):
        with output:
            clear_output()
            submit_to_samatrix(token_box.value)

    submit_button.on_click(_on_submit_clicked)

    display(token_box)
    display(submit_button)
    display(output)

except Exception as exc:
    print("Interactive button could not be loaded in this notebook environment.")
    print("Use this fallback instead:")
    print("from getpass import getpass")
    print("submit_to_samatrix(getpass('Paste Colab Submission Token: '))")
    print("Widget error:", exc)
